In [0]:
%run ../../02_common_utils/raw_to_landing

In [0]:
log_pipeline_message(spark, "unknown", 'INFO', 'landing_customer_all', f'Starting landing extraction for Customer domain (batch: {carried_batch})')
start_pipeline_run(spark, "unknown", 1)
log_domain_run_status(spark, "unknown", 1, 'CUSTOMER', 'RUNNING')

In [0]:
recon_df=load_domain("customer")

In [0]:
# recon_df.display()

In [0]:
# spark.read.parquet("/Volumes/charles_schwab_retailbrokerage_dev_team_lemma/landing/pwg/Batch1/customermgmt/").limit(10).display()

In [0]:
from pyspark.sql.functions import col, count, when

# Assuming target_base_path is still defined in your notebook
# target_base_path = f"/Volumes/charles_schwab_retailbrokerage_dev_{team_name}/landing/landing_{team_name}/"
source_base_path = "abfss://raw@schwabdldevsa.dfs.core.windows.net/"
target_base_path = f"/Volumes/charles_schwab_retailbrokerage_dev_team_lemma/landing/pwg/"
successful_ingestions = [
    ("Batch1", "Prospect"),
    ("Batch1", "WatchHistory"),
    ("Batch1", "StatusType"),
    ("Batch1", "TaxRate"),
    ("Batch1", "CustomerMgmt"),
    ("Batch2", "Customer"),
    ("Batch2", "Prospect"),
    ("Batch2", "WatchHistory"),
    ("Batch3", "Customer"),
    ("Batch3", "Prospect"),
    ("Batch3", "WatchHistory")
]

print("--- 🔍 Scanning for All-Null Columns ---\n")

for batch, table_name in successful_ingestions:
    parquet_path = f"{target_base_path}{batch}/{table_name.lower()}"
    
    try:
        # 1. Load the dataframe
        df = spark.read.format("parquet").load(parquet_path)
        total_rows = df.count()
        
        if total_rows == 0:
            print(f"⚠️ {batch} -> {table_name}: Table is completely empty (0 rows).")
            continue
            
        # 2. Build a list of expressions to count nulls in every column
        # This creates a single row with the sum of nulls for each column
        null_count_exprs = [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
        
        # 3. Execute the count and convert the single row result to a Python dictionary
        null_counts_dict = df.select(*null_count_exprs).collect()[0].asDict()
        
        # 4. Filter for columns where the null count exactly matches the total row count
        all_null_columns = [c for c, null_count in null_counts_dict.items() if null_count == total_rows]
        
        # 5. Print the results
        if all_null_columns:
            print(f"❌ {batch} -> {table_name}: Found {len(all_null_columns)} ALL-NULL column(s):")
            print(f"    ↳ {', '.join(all_null_columns)}\n")
        else:
            print(f"✅ {batch} -> {table_name}: Looks good! No all-null columns.\n")
            
    except Exception as e:
        print(f"⚠️ Could not process {table_name} in {batch}. Error: {e}\n")
        
print("--- Scan Complete ---")

In [0]:
log_domain_run_status(spark,"unknown", 1, 'CUSTOMER', 'COMPLETED')
end_pipeline_run(spark, "unknown", 'SUCCESS')
log_pipeline_message(spark, "unknown", 'INFO', 'landing_customer_all', 'Successfully completed landing ingestion for customer domain.')